# 14 Transformer Token Classification — CoNLL-2003


Fine-tuning `bert-base-cased` cho NER. Strategy A được dùng trong loss; checkpoint và dự đoán được lưu để tái sử dụng.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLP/lab09'
import os, json, random, numpy as np
os.makedirs(DRIVE_DIR, exist_ok=True)
SEED = 42
random.seed(SEED); np.random.seed(SEED)
!pip -q install datasets==3.6.0 transformers
import torch, time
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from torch.utils.data import DataLoader
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)


Mounted at /content/drive
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


In [ ]:
ds=load_dataset('eriktks/conll2003', trust_remote_code=True)
label_names=ds['train'].features['ner_tags'].feature.names
label2id={x:i for i,x in enumerate(label_names)}; id2label={i:x for x,i in label2id.items()}
checkpoint='bert-base-cased'
tokenizer=AutoTokenizer.from_pretrained(checkpoint, use_fast=True)
model=AutoModelForTokenClassification.from_pretrained(checkpoint,num_labels=len(label_names),id2label=id2label,label2id=label2id)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); model.to(device)
print('Device:',device,'| train:',len(ds['train']),'| test:',len(ds['test']))
def encode_split(split):
    result=[]
    for row in split:
        enc=tokenizer(row['tokens'],is_split_into_words=True,truncation=True,max_length=256)
        word_ids=enc.word_ids(); labels=[]; previous=None
        for wid in word_ids:
            if wid is None or wid==previous: labels.append(-100)
            else: labels.append(int(row['ner_tags'][wid]))
            previous=wid
        result.append({'input_ids':enc['input_ids'],'attention_mask':enc['attention_mask'],'labels':labels})
    return result
train_data=encode_split(ds['train']); test_data=encode_split(ds['test'])
def collate(batch):
    features=[{'input_ids':x['input_ids'],'attention_mask':x['attention_mask']} for x in batch]
    padded=tokenizer.pad(features,return_tensors='pt')
    y=torch.full_like(padded['input_ids'],-100)
    for i,x in enumerate(batch): y[i,:len(x['labels'])]=torch.tensor(x['labels'])
    padded['labels']=y; return padded
generator=torch.Generator().manual_seed(SEED)
loader=DataLoader(train_data,batch_size=16,shuffle=True,generator=generator,collate_fn=collate)
optimizer=torch.optim.AdamW(model.parameters(),lr=3e-5)
for epoch in range(2):
    model.train(); total=0
    for step,batch in enumerate(loader,1):
        batch={k:v.to(device) for k,v in batch.items()}
        optimizer.zero_grad(); loss=model(**batch).loss; loss.backward(); optimizer.step(); total+=loss.item()
        if step%200==0: print(f'epoch {epoch+1} step {step}/{len(loader)} loss {loss.item():.4f}')
    print(f'epoch {epoch+1} mean loss {total/len(loader):.4f}')


README.md:   0%|          | 0.00/12.3k [00:00<?, ?B/s]

conll2003.py:   0%|          | 0.00/9.57k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

... (4 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

Device: cpu | train: 14041 | test: 3453
epoch 1 step 200/878 loss 0.1095
epoch 1 step 400/878 loss 0.0414
epoch 1 step 600/878 loss 0.0099


In [ ]:
import os, json, numpy as np
model_dir=f'{DRIVE_DIR}/bert-base-cased-conll2003'
model.save_pretrained(model_dir); tokenizer.save_pretrained(model_dir)
model.eval(); prediction_rows=[]; test_rows=[]
with torch.no_grad():
    for row in ds['test']:
        enc=tokenizer(row['tokens'],is_split_into_words=True,truncation=True,max_length=256,return_tensors='pt')
        word_ids=tokenizer(row['tokens'],is_split_into_words=True,truncation=True,max_length=256).word_ids()
        inputs={k:v.to(device) for k,v in enc.items()}; logits=model(**inputs).logits[0]
        token_preds=logits.argmax(-1).cpu().tolist(); word_preds=[]; seen=set()
        for wid,p in zip(word_ids,token_preds):
            if wid is not None and wid not in seen: word_preds.append(label_names[p]); seen.add(wid)
        word_preds=word_preds[:len(row['tokens'])]+['O']*max(0,len(row['tokens'])-len(word_preds))
        prediction_rows.append({'words':row['tokens'],'pred':word_preds})
        test_rows.append({'words':row['tokens'],'gold':[label_names[i] for i in row['ner_tags']]})
with open(f'{DRIVE_DIR}/transformer_predictions.json','w',encoding='utf-8') as f: json.dump(prediction_rows,f,ensure_ascii=False)
with open(f'{DRIVE_DIR}/test_gold.json','w',encoding='utf-8') as f: json.dump(test_rows,f,ensure_ascii=False)
print('Saved model and',len(prediction_rows),'test predictions')
